# BorrowBox V2.1 — Seed Dataset & Database Initialization Strategy

**Status:** Final implementation planning baseline  
**Scope:** Fresh V2.1 database only  
**V1 migration:** Out of scope

## 1. Purpose

V2.1 starts with a fresh database.

The seed dataset is deterministic development data designed to exercise the V2.1 domain model:

```text
User
Community
Membership
Community role/context
Asset
AssetUnit
CommunityListing
```

The seed exists to test the architecture, not merely to populate the UI.

---

## 2. Seed users

Use predictable fictional development users:

| User | Purpose |
|---|---|
| Ahmed | Manager + multi-unit asset owner |
| Salah | Community member |
| Omar | Manager + asset owner |
| Youssef | Member + asset owner |
| Karim | Member + asset owner |

Use development-only authentication values. Never seed real passwords, private addresses, or real personal information.

---

## 3. Seed communities

Create:

| Community | Type | Purpose |
|---|---|---|
| CSE Department | COLLEGE | Academic community |
| Hostel Block B | HOSTEL | Residential community |
| Engineering Office | OFFICE | Workplace community |

Each has:

```text
name
description
type
status = ACTIVE
created_by
location fields
created_at
updated_at
```

Development locations are fictional/test values.

### Community-name rule

For V2.1:

> The same creator/manager cannot create two ACTIVE communities with the same name.

Different creators may use the same community name.

Therefore the seed natural key for a community is:

```text
(created_by, name)
```

Do not use `name` alone as the unique seed identity.

---

## 4. Seed memberships

Use community-specific membership context.

```text
Ahmed
  → CSE Department
  → MANAGER
  → {"program":"CSE","year":4,"section":"A"}

Salah
  → CSE Department
  → MEMBER
  → {"program":"CSE","year":4,"section":"A"}

Omar
  → Hostel Block B
  → MANAGER
  → {"block":"B","floor":3,"room":"B-302"}

Youssef
  → Hostel Block B
  → MEMBER
  → {"block":"B","floor":3,"room":"B-304"}

Ahmed
  → Engineering Office
  → MEMBER
  → {"department":"Engineering","team":"Platform"}

Karim
  → Engineering Office
  → MEMBER
  → {"department":"Engineering","team":"QA"}
```

Store context in:

```text
Membership.context_metadata JSON
```

Membership rows must be inserted only after users and communities exist.

---

## 5. Seed assets

Create assets with both multi-unit and single-unit ownership.

### AHMED_FOOTBALL

```text
Owner = Ahmed
Physical units = 2
```

### OMAR_CAMERA

```text
Owner = Omar
Physical units = 1
```

### YOUSSEF_DRILL

```text
Owner = Youssef
Physical units = 1
```

### KARIM_CALCULATOR

```text
Owner = Karim
Physical units = 1
```

### KARIM_SPARE_LAPTOP

```text
Owner = Karim
Physical units = 1
```

The stable development identity must not depend on title alone.

---

## 6. AssetUnit creation rule

When an asset with quantity `N` is created, the backend must perform the following as ONE atomic database transaction:

```text
BEGIN TRANSACTION
    ↓
insert 1 Asset
    ↓
insert exactly N AssetUnits
    ↓
COMMIT
```

For `AHMED_FOOTBALL`:

```text
Asset
 ├── AssetUnit #1
 └── AssetUnit #2
```

There is no aggregate-first or delayed-unit creation model.

The Asset has no `total_quantity` column. Quantity is derived from AssetUnits.

---

## 7. Community listings

The same physical asset can be offered in multiple communities.

### AHMED_FOOTBALL

Create exactly three CommunityListings:

```text
CSE Department
Hostel Block B
Engineering Office
```

Do NOT create three Football Assets.

### OMAR_CAMERA

```text
Hostel Block B
```

### YOUSSEF_DRILL

```text
Hostel Block B
CSE Department
```

### KARIM_CALCULATOR

```text
Engineering Office
```

### KARIM_SPARE_LAPTOP

```text
No CommunityListing
```

The Spare Laptop exists in owner inventory but is invisible to community Explore.

---

## 8. Listing authorization

A CommunityListing may be created or activated only when:

```text
Asset.owner_id
    ↓
ACTIVE Membership
    ↓
same Community
```

The backend service/transaction layer must enforce this.

The database separately enforces:

```text
CommunityListing.asset_id → Asset.id
CommunityListing.community_id → Community.id
```

Seed setup must include a negative authorization test:

```text
Karim
  → Engineering Office ✅
  → CSE Department ❌
```

Attempting to list Karim's asset in CSE Department must fail server-side.

---

## 9. Public Explore/search response

V2.1 public Explore/search must return:

```text
one Asset summary per CommunityListing
```

A summary can contain:

```text
assetId
title
description
category
listingStatus
totalUnits
availableUnits
borrowedUnits
```

Do not expose:

```text
AssetUnit IDs
internal seed identities
internal reservation identifiers
```

AssetUnit rows remain an internal physical-inventory representation.

---

## 10. Cross-community shared availability fixture

V2.1 does not yet implement the Transaction engine.

To make the shared-inventory invariant immediately testable, the deterministic seed state is:

```text
AHMED_FOOTBALL
    Unit #1 → AVAILABLE
    Unit #2 → BORROWED
```

Therefore all three CommunityListings must return:

```text
totalUnits = 2
availableUnits = 1
borrowedUnits = 1
```

This must be identical in:

```text
CSE Department
Hostel Block B
Engineering Office
```

### Important fixture rule

`Unit #2 = BORROWED` is a **V2.1 development fixture only**.

It does not represent a real borrower, loan, handover, or Transaction.

When V2.2 introduces Transactions, the seed must be updated so every BORROWED AssetUnit is associated with a real active Transaction.

### Dynamic aggregate test

A V2.1 test/fixture must be able to change:

```text
Unit #1:
AVAILABLE → BORROWED
```

and verify:

```text
totalUnits = 2
availableUnits = 0
borrowedUnits = 2
```

across all three listings.

Changing a unit from BORROWED back to AVAILABLE must restore:

```text
availableUnits = 1
borrowedUnits = 1
```

This verifies that community listings never own separate quantities.

---

## 11. Initialization order

A clean reset follows:

```text
stop application
    ↓
remove V2 database/volume when requested
    ↓
start MySQL
    ↓
create V2.1 schema
    ↓
seed data
    ↓
start backend
    ↓
start frontend
```

### LOCKED foreign-key insertion order

Seed data must be inserted in exactly this dependency order:

```text
1. users
2. communities
3. memberships
4. assets
5. asset_units
6. community_listings
```

Reason:

```text
communities.created_by → users.id

memberships.user_id → users.id
memberships.community_id → communities.id

assets.owner_id → users.id

asset_units.asset_id → assets.id

community_listings.asset_id → assets.id
community_listings.community_id → communities.id
```

Never insert a dependent row before its referenced parent row exists.

---

## 12. Idempotent seed strategy

Restarting the application must not duplicate seed records.

### LOCKED natural/stable identities

Use:

```text
users:
    email

communities:
    (created_by, name)

memberships:
    (user_id, community_id)

assets:
    stable development seed identity

asset_units:
    deterministic unit identity within the parent Asset

community_listings:
    (asset_id, community_id)
```

For raw SQL, use an idempotent pattern such as:

```sql
INSERT ... ON DUPLICATE KEY UPDATE ...
```

or a guarded existence check based on the natural/stable key.

For a Spring seed component:

```text
resolve deterministic key
    ↓
if present → reuse
if absent  → create
```

The seed must be safe to execute repeatedly against the same V2 development database.

### Important

Do not make seed idempotency depend on:

```text
title alone
community name alone
database auto-increment IDs alone
```

---

## 13. Deterministic Asset identity

Asset titles are editable and not unique.

Use stable seed identities such as:

```text
AHMED_FOOTBALL
OMAR_CAMERA
YOUSSEF_DRILL
KARIM_CALCULATOR
KARIM_SPARE_LAPTOP
```

These can be implemented as:

```text
dedicated development seed key
```

or another stable internal identifier.

The exact mechanism must not leak this seed key into the public API.

---

## 14. Unlisted Asset scenario

`KARIM_SPARE_LAPTOP` must demonstrate:

```text
Asset exists
AssetUnit exists
CommunityListing does not exist
```

Expected behavior:

```text
Owner inventory → visible
Community Explore → invisible
```

---

## 15. Repeatable reset

A clean reset must reproduce the same logical dataset every time.

Expected baseline:

```text
Communities = 3

Logical Assets = 5

AssetUnits:
  Football      2
  Camera        1
  Drill         1
  Calculator    1
  Spare Laptop  1

Total AssetUnits = 6

CommunityListings:
  Football      3
  Camera        1
  Drill         2
  Calculator    1
  Spare Laptop  0

Total CommunityListings = 7
```

The exact user/membership counts may evolve, but these domain scenarios must remain.

---

## 16. Seed data is development-only

Seed content must never contain:

```text
real credentials
real addresses
real personal data
real private location information
```

Use fictional development values.

---

## 17. V2.1 scope boundary

The baseline seed does NOT create:

```text
Transaction
BorrowRequest
BorrowRecord
Handover
Evidence
Notifications
Reputation
Flags
AI
Tribunal
```

The single seeded BORROWED AssetUnit is explicitly a physical-state fixture for testing shared availability.

---

## 18. Acceptance scenarios

### A. Community visibility

A user can see only communities for which they have the appropriate membership.

### B. Selective visibility

Ahmed's Football appears only in its three listings.

### C. Shared inventory

Changing one Football unit's state changes availability in all three communities.

### D. Unlisted ownership

The Spare Laptop appears in owner inventory but nowhere in Explore.

### E. Authorization

A non-member cannot create/activate an asset listing in that community.

### F. Aggregate public API

Explore returns one summary per listing and hides AssetUnit IDs.

### G. Idempotency

Restarting the app does not create duplicate seed data.

### H. Reset

A clean reset recreates the same logical baseline.

### I. Foreign-key safety

Seed initialization succeeds because rows are inserted in dependency order.

---

## 19. Final initialization checklist

```text
[ ] V2 uses a fresh database.
[ ] V1 migration is not part of the runtime.
[ ] users are inserted first.
[ ] communities are inserted second.
[ ] memberships are inserted third.
[ ] assets are inserted fourth.
[ ] asset_units are inserted fifth.
[ ] community_listings are inserted last.

[ ] Seed initialization is idempotent.
[ ] User key = email.
[ ] Community seed key = (created_by, name).
[ ] Membership key = (user_id, community_id).
[ ] Asset seed identity is deterministic.
[ ] AssetUnit identity is deterministic within Asset.
[ ] Listing key = (asset_id, community_id).

[ ] AHMED_FOOTBALL has exactly two AssetUnits.
[ ] Unit #1 is AVAILABLE.
[ ] Unit #2 is BORROWED as a V2.1 fixture only.
[ ] All three Football listings show 2 total / 1 available / 1 borrowed.
[ ] Public Explore exposes aggregate counts only.
[ ] Spare Laptop is unlisted.
[ ] Unauthorized listing attempt fails server-side.
[ ] Restarting the application does not duplicate seed data.
[ ] Clean reset reproduces the baseline.
```

## 20. Core principle

> **The seed dataset must exercise real V2.1 domain invariants. It must not hide architectural mistakes by using arbitrary demo records.**


## 21. Locked implementation mechanism

The V2.1 development seed is implemented as a Spring `ApplicationRunner` (or equivalent startup initializer) controlled by:

```text
borrowbox.seed.enabled=true
```

The seed must be:

```text
idempotent
deterministic
development-only
safe to rerun
```

Schema creation is handled separately by the version-controlled V2 `schema.sql` baseline with:

```text
spring.jpa.hibernate.ddl-auto=validate
```

The initializer must not read or transform V1 tables.

## 22. Admission fixtures

The deterministic seed should include at least:

```text
one MANAGER_APPROVAL community
one LOCATION_VERIFIED community
```

and enough memberships to verify both admission modes.

The seed remains a fixture only. It does not create real-world location tracking or real personal data.
